# FLATUP GYM ポーズ画像ジェネレーター(Google Colab用)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatup1/openqlow/blob/main/girl-power-op/colab_generate_poses.ipynb)

`chara.png`(キャラの基準画像)を参照しながら、アニメに必要なポーズ画像を
**無料のColab GPU** で生成するノートブックです。すべてブラウザだけで完結します。

## 使い方(3ステップ)
1. GPUは自動で選ばれます(もし「GPUがない」と出たら: メニュー **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んで保存)
2. 上から順にセルを実行(▶ボタン)。途中で `chara.png` のアップロードを求められます
3. 最後のセルで `poses.zip` がダウンロードされる → 中身を `girl-power-op/poses/` に入れて `npm run video`

生成された画像はこのページ内にその場で表示されるので、気に入らなければ
`SEED` の数字を変えて何度でも作り直せます。

In [ ]:
# ① GPUが使えるか確認(「Tesla T4」などと出ればOK)
!nvidia-smi -L

In [ ]:
# ② 必要な部品を入れる(1〜2分)
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
# ③ キャラの基準画像(chara.png)をアップロード
from google.colab import files
from PIL import Image
import io

up = files.upload()  # ファイル選択ダイアログが出ます
ref_image = Image.open(io.BytesIO(list(up.values())[0])).convert('RGB')
display(ref_image.resize((256, int(256 * ref_image.height / ref_image.width))))
print('この画像を参照してポーズを作ります')

In [ ]:
# ④ 画像生成AI(SDXL)+ 参照画像アダプター(IP-Adapter)を読み込む(数分)
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
)
# IP-Adapter: 「この画像と同じキャラで」と伝えるための部品
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models',
                     weight_name='ip-adapter_sdxl.bin')
pipe.set_ip_adapter_scale(0.7)   # 参照画像にどれだけ似せるか(0.5〜0.9で調整)
pipe.enable_model_cpu_offload()  # 無料GPUのメモリでも動くように
print('準備OK')

In [ ]:
# ⑤ ポーズを生成する(1枚あたり1〜2分)
import os
os.makedirs('poses', exist_ok=True)

SEED = 42  # 数字を変えると別の結果になる(気に入るまで変えてOK)

# 全ポーズ共通のスタイル指定(FLATUP GYMの共通ルール)
STYLE = ('warm 3D animated toy-like movie style, soft cinematic lighting, '
         'friendly expression, cute original chibi gym mascot, full body, '
         'vertical composition, character centered, feet at the bottom, '
         'consistent outfit and hairstyle and colors, clean bright gym background')
NEGATIVE = ('scary, aggressive, angry, violent, injury, text, watermark, logo, '
            'extra limbs, deformed hands, lowres, blurry')

POSES = {
    'idle':   'standing in a relaxed kickboxing guard pose',
    'punch1': 'throwing a small left punch, dynamic but friendly',
    'punch2': 'throwing a small right punch, energetic and joyful',
    'jump':   'jumping happily in the air, arms up',
    'win':    'cheerful victory pose, smiling with confidence',
    # 感情ポーズも欲しければ、ここに追加(ファイル名: 説明文):
    # 'fall': 'sitting on the gym mat after a harmless failed punch, embarrassed but charming',
}

for name, action in POSES.items():
    g = torch.Generator('cuda').manual_seed(SEED)
    img = pipe(
        prompt=f'{action}, {STYLE}',
        negative_prompt=NEGATIVE,
        ip_adapter_image=ref_image,
        num_inference_steps=30,
        width=832, height=1216,   # 縦長(chara.pngと同じ比率に近い)
        generator=g,
    ).images[0]
    img.save(f'poses/{name}.png')
    print(f'--- {name}.png ---')
    display(img.resize((208, 304)))  # その場でブラウザ確認

In [ ]:
# ⑥ まとめてダウンロード
!zip -q -r poses.zip poses
from google.colab import files
files.download('poses.zip')

## つぎにやること

1. `poses.zip` を解凍して、中の画像を `girl-power-op/poses/` に入れる
2. `npm run video` → キャラがポーズを切りかえる本物のアニメになる
3. ブラウザで確認したいだけなら `index.html?loop=1` を開けばループ再生される

## うまくいかないとき

- **キャラが似ていない** → ④の `set_ip_adapter_scale` を `0.8〜0.9` に上げる
- **ポーズが変わらない** → 逆に `0.5〜0.6` に下げる(似せる力を弱めるとポーズが自由になる)
- **手や足が変** → ⑤の `SEED` を変えて引き直す(ガチャと同じ)
- **メモリ不足エラー** → ランタイム → セッションを再起動 して②から実行し直す